# Foodpanda APAC Customer Feedback Analysis v2 — Aspect Extraction Pipeline

**Author:** Junior — Data Analyst Portfolio Project

This notebook runs the production extraction layer for v2 of the foodpanda APAC customer feedback analysis. It sends ~272k customer reviews through Google's Gemini 2.5 Flash Lite to perform aspect-based sentiment analysis (ABSA) — identifying *what* customers are complaining or praising about, not just whether they're happy.

[**v1**](foodpanda_apac/foodpanda_APAC_analysis.ipynb) used unigram keyword frequency to surface patterns across 4.6M reviews and 11 markets. It answered *"what words come up most?"* — useful for a first pass, but unable to distinguish between a review mentioning "bad" as a complaint versus "not bad" as praise.

**v2** adds an AI extraction layer that reads each review and labels specific aspects (rider service, delivery time, handling, order accuracy, packaging, vendor quality flags) with sentiment and quoted evidence. The architecture is hybrid: the LLM handles qualitative interpretation; deterministic Python handles all aggregation, filtering, and output downstream.

**Scope:** 9 active APAC markets (foodpanda exited Thailand in May 2025), latest 3 months of reviews (November 2025 – January 2026), reviews with 20+ characters of text.

**Why Gemini Flash Lite?** It offered the balance this project needed — low cost per call, fast enough for iterative prompt development, and sufficient output quality for structured aspect extraction, as confirmed by the validation results.

---

## 1. Setup and Configuration

Standard imports plus the `google-genai` SDK for Gemini API access. The API key is read from an environment variable rather than hardcoded — this notebook is published to GitHub, so secrets stay out of version control. The output directory holds one JSON file per market, which the downstream analysis notebook will consume.

In [1]:
import pandas as pd
import json
import asyncio
import time
import os
from pathlib import Path
from google import genai
from google.genai import types

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found. Set it with: export GEMINI_API_KEY='your-key-here'"
    )

client = genai.Client(api_key=GEMINI_API_KEY)

CSV_PATH = '/Users/juniork/Desktop/Case study/Foodpanda Analysis/foodpanda_APAC_final.csv'
OUTPUT_DIR = Path('/Users/juniork/Desktop/Case study/Foodpanda Analysis/v2_production/')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL = 'gemini-2.5-flash-lite'

print(f"Output directory: {OUTPUT_DIR}")
print(f"Model: {MODEL}")

Output directory: /Users/juniork/Desktop/Case study/Foodpanda Analysis/v2_production
Model: gemini-2.5-flash-lite


Quick connectivity check — if the API key is invalid or the model name has changed, this fails immediately rather than 2 hours into an extraction run.

In [2]:
test_response = client.models.generate_content(
    model=MODEL,
    contents='Return the JSON array: [{"status": "ok"}]',
    config=types.GenerateContentConfig(
        temperature=0.0,
        response_mime_type='application/json'
    )
)
print("API test:", test_response.text)

API test: [{"status": "ok"}]


Gemini API latency varies by time of day and server load.
This test establishes a baseline before committing to an extraction run — if average latency is above 5 seconds, it may be worth waiting for a quieter window.

In [8]:
async def latency_test():
    times = []
    for i in range(5):
        t0 = time.time()
        await client.aio.models.generate_content(
            model=MODEL,
            contents="Return JSON: []",
            config=types.GenerateContentConfig(
                temperature=0.0,
                response_mime_type='application/json'
            )
        )
        elapsed = time.time() - t0
        times.append(elapsed)
        print(f"  Call {i+1}: {elapsed:.2f}s")

    avg = sum(times) / len(times)
    print(f"\n  Average: {avg:.2f}s per call")
    if avg < 2:
        print("  → API is fast. Expected throughput: ~5-10 reviews/sec with concurrency=10.")
    elif avg < 5:
        print("  → API is moderate. Expected throughput: ~2-5 reviews/sec with concurrency=10.")
    else:
        print("  → API is slow (Google-side). Expected throughput: ~1-2 reviews/sec. Consider waiting for a quieter window.")

await latency_test()

  Call 1: 1.28s
  Call 2: 0.84s
  Call 3: 0.80s
  Call 4: 1.17s
  Call 5: 3.18s

  Average: 1.45s per call
  → API is fast. Expected throughput: ~5-10 reviews/sec with concurrency=10.


---

## 2. The Locked Extraction Prompt (V6)

This prompt went through 5 iterations during development (V1–V5 are in the exploration notebook). V6 is the locked production version — no further changes will be made. Iterating against the validation set after locking would cause data leakage, so the prompt was frozen before final validation scoring.

The prompt defines a 6-aspect taxonomy covering issues that foodpanda's platform can act on:

| Aspect | What it captures |
|---|---|
| `rider_service` | Rider attitude, behaviour, communication, navigation |
| `delivery_time` | Explicit mentions of speed — late, slow, fast, on time |
| `delivery_handling` | Food condition damaged in transit — spills, squashed items |
| `order_accuracy` | Missing or wrong items |
| `packaging` | Container failures, missing utensils |
| `vendor_quality_flag` | Severe food safety issues only (poisoning, foreign objects, hygiene) — flagged for restaurant audit |

The taxonomy deliberately excludes normal restaurant quality complaints (bland, small portions, overcooked). These are vendor-side issues that foodpanda cannot directly control — they would add noise without actionability. The `vendor_quality_flag` is the exception: severe food safety issues warrant platform-level investigation regardless of vendor responsibility.

In [4]:
PROMPT_V6 = """You are analyzing a food delivery company customer review to identify food-delivery-controllable issues.

Use ONLY these 6 aspect labels:

1. rider_service
   - Use for: rider attitude, behavior, communication, finding address
   - DO NOT use for: rider being thanked generally without specific behavior
   - Example YES: "rider was rude", "delivery man yelled", "rider couldn't find address", "rider was polite"
   - Example NO: "thanks to rider" with no other context

2. delivery_time
   - Use for: EXPLICIT mention of speed (late, slow, fast, on time, prompt)
   - DO NOT use for: "still warm" (that's handling, not time)
   - Example YES negative: "took 2 hours", "very long delivery", "1.45pm only received"
   - Example YES positive: "very prompt delivery", "fast delivery", "arrived in 15 mins"
   - Example NO: "food was hot", "satisfying meal"

3. delivery_handling
   - Use for: food condition damaged in transit (spilled, squashed)
   - Example YES: "items arrived with spillage", "soup leaked everywhere"
   - Example NO: pure food taste complaints

4. order_accuracy
   - Use for: EXPLICIT missing items or EXPLICIT wrong items
   - DO NOT use for: food being bad quality, small portions, or general complaints
   - Example YES: "missing 2 eggs", "ordered roasted got steamed", "wrong item"
   - Example NO: "food was bland", "portion small", "rice undercooked"

5. packaging
   - Use for: container failures (leaking, broken, missing utensils)
   - Example YES: "container was leaking", "no chopsticks provided"
   - Example NO: food taste or quality issues

6. vendor_quality_flag
   - Use ONLY for SEVERE food issues warranting restaurant investigation:
     * Food poisoning symptoms (vomit, diarrhoea, sick)
     * Foreign objects (hair, plastic, insects in food)
     * Hygiene failures (uncleaned ingredients, mold, off smell)
     * Explicit safety concerns
   - DO NOT use for: bland taste, small portions, burnt food, dry/tough texture, undercooked
     (these are normal restaurant quality issues, NOT audit triggers)
   - Example YES: "threw up after eating", "found hair in food", "intestine not cleaned"
   - Example NO: "too dry", "burnt", "shrunken meat", "bland", "small portion"

For each aspect mentioned, return:
- "aspect": exactly one label from the 6 above
- "sentiment": exactly "positive", "negative", or "neutral"
- "evidence": direct quote from review (5-20 words)

STRICT RULES:
1. If the review is purely about food taste, texture, portion, or non-severe quality, output [].
2. Each aspect requires EXPLICIT textual evidence. Do not infer aspects from star rating alone.
3. Output ONLY valid JSON array. No prose.

Review (rating: {rating} stars):
{text}

JSON output:"""

---

## 3. Production Extraction Function

This is the core of the pipeline. The function processes one market at a time with built-in resilience:

1. **Load and filter** — reads the master CSV, filters to the target country and the 3-month window (Nov 2025 – Jan 2026), drops reviews shorter than 20 characters (these are typically emoji-only or "ok" and yield no extractable aspects).
2. **Resume from checkpoint** — if a partial output file already exists, the function picks up where it left off. This is critical for large markets like Pakistan (~125k reviews) where interruptions are inevitable.
3. **Async extraction** — each batch of 500 reviews is dispatched as async coroutines via `client.aio.models.generate_content`, with an `asyncio.Semaphore` capping concurrency at 10 simultaneous API calls. This achieves true parallelism — unlike `ThreadPoolExecutor`, the async SDK releases the event loop while waiting for API responses.
4. **Checkpoint every 500 reviews** — progress is saved to disk after each batch completes. Results are sorted by review index before saving since async completion order is non-deterministic. Worst case on a crash: 500 reviews of rework.

The function returns no analysis — it's a pure extraction step. All interpretation happens in the downstream analysis notebook. This separation keeps the LLM's role narrow (read review → label aspects) and the analyst's role clear (aggregate → interpret → recommend).

In [5]:
def parse_gemini_json(raw_text):
    """Parse JSON from Gemini output, handling trailing garbage (e.g. []\\n[], []<>)."""
    try:
        return json.loads(raw_text), 'ok'
    except json.JSONDecodeError as e:
        if e.msg == 'Extra data' and e.pos <= 3:
            try:
                return json.loads(raw_text[:e.pos]), 'ok_truncated'
            except json.JSONDecodeError:
                pass
        return None, f"json_error: {e}"


async def extract_market(country_name, batch_size=500, max_retries=3, concurrency=10):
    """Extract aspects from all reviews for one market. Checkpoints every batch_size reviews."""

    # --- Load and filter ---
    df = pd.read_csv(CSV_PATH, low_memory=False)
    df['createdAt'] = pd.to_datetime(df['createdAt'])
    market = df[
        (df['country'] == country_name)
        & (df['createdAt'] >= '2025-11-01')
        & (df['createdAt'] <= '2026-01-31')
    ].copy()
    market = market[market['text'].astype(str).str.len() >= 20].reset_index(drop=True)

    output_file = OUTPUT_DIR / f"{country_name.lower().replace(' ', '_')}_aspects.json"
    print(f"\n{'='*60}")
    print(f"Market: {country_name}")
    print(f"Reviews after filtering: {len(market):,}")
    print(f"Output: {output_file}")

    # --- Resume from checkpoint ---
    results = []
    start_idx = 0
    if output_file.exists():
        with open(output_file) as f:
            results = json.load(f)
        start_idx = len(results)
        print(f"Resuming from checkpoint: {start_idx:,} reviews already processed")

    if start_idx >= len(market):
        print("Already complete.")
        return results

    print(f"Processing reviews {start_idx:,} to {len(market):,} (concurrency={concurrency})...")
    print(f"{'='*60}\n")

    t_start = time.time()
    semaphore = asyncio.Semaphore(concurrency)

    async def process_review(idx):
        row = market.iloc[idx]
        prompt = PROMPT_V6.format(rating=int(row['overall']), text=str(row['text']))

        parsed = None
        parse_status = 'ok'

        async with semaphore:
            for attempt in range(max_retries):
                try:
                    response = await client.aio.models.generate_content(
                        model=MODEL,
                        contents=prompt,
                        config=types.GenerateContentConfig(
                            temperature=0.0,
                            response_mime_type='application/json'
                        )
                    )
                    parsed, parse_status = parse_gemini_json(response.text)
                    if parsed is not None:
                        break
                    break
                except Exception as e:
                    if attempt < max_retries - 1:
                        await asyncio.sleep(2 ** attempt)  # 1s, 2s, 4s
                    else:
                        parsed = None
                        parse_status = f"api_error: {e}"

        return {
            'review_idx': idx,
            'storeId': row.get('StoreId', None),
            'country': country_name,
            'rating': int(row['overall']),
            'text': str(row['text']),
            'createdAt': str(row['createdAt']),
            'parsed': parsed,
            'parse_status': parse_status,
            'rider': row.get('rider', None),
            'restaurant_food': row.get('restaurant_food', None)
        }

    # --- Extract in batches with async concurrency ---
    for batch_start in range(start_idx, len(market), batch_size):
        batch_end = min(batch_start + batch_size, len(market))

        tasks = [process_review(idx) for idx in range(batch_start, batch_end)]
        batch_results = await asyncio.gather(*tasks)

        batch_results = sorted(batch_results, key=lambda r: r['review_idx'])
        results.extend(batch_results)

        with open(output_file, 'w') as f:
            json.dump(results, f)

        elapsed = time.time() - t_start
        processed = batch_end - start_idx
        rate = processed / elapsed
        remaining = (len(market) - batch_end) / rate if rate > 0 else 0
        errors_so_far = sum(1 for r in results if r['parse_status'] not in ('ok', 'ok_truncated'))
        print(
            f"  [{batch_end:,}/{len(market):,}] "
            f"checkpoint saved | "
            f"{rate:.1f} reviews/sec | "
            f"~{remaining/60:.0f} min remaining | "
            f"{errors_so_far} errors"
        )

    # --- Final summary ---
    elapsed = time.time() - t_start
    errors = sum(1 for r in results if r['parse_status'] not in ('ok', 'ok_truncated'))
    truncated = sum(1 for r in results if r['parse_status'] == 'ok_truncated')
    print(f"\nDone: {country_name}")
    print(f"  Total: {len(results):,} reviews")
    print(f"  Truncated recoveries: {truncated:,}")
    print(f"  Errors: {errors:,} ({errors/len(results)*100:.1f}%)")
    print(f"  Time: {elapsed/60:.1f} min")
    print(f"  Saved to: {output_file}")

    return results

---

## 4. Extraction Results

All 9 active APAC markets were extracted using the V6 prompt. Singapore ran first as the validation market, followed by the remaining 8 in order of availability. Each cell below shows the extraction call and its output — review count, truncated recoveries (the `[]\n[]` pattern handled by `parse_gemini_json`), residual errors, and wall-clock time.

In [9]:
sg_results = await extract_market('Singapore')


Market: Singapore
Reviews after filtering: 9,813
Output: /Users/juniork/Desktop/Case study/Foodpanda Analysis/v2_production/singapore_aspects.json
Processing reviews 0 to 9,813 (concurrency=10)...

  [500/9,813] checkpoint saved | 10.6 reviews/sec | ~15 min remaining | 0 errors
  [1,000/9,813] checkpoint saved | 10.1 reviews/sec | ~15 min remaining | 0 errors
  [1,500/9,813] checkpoint saved | 8.6 reviews/sec | ~16 min remaining | 0 errors
  [2,000/9,813] checkpoint saved | 8.9 reviews/sec | ~15 min remaining | 0 errors
  [2,500/9,813] checkpoint saved | 9.1 reviews/sec | ~13 min remaining | 0 errors
  [3,000/9,813] checkpoint saved | 9.1 reviews/sec | ~12 min remaining | 0 errors
  [3,500/9,813] checkpoint saved | 9.2 reviews/sec | ~11 min remaining | 0 errors
  [4,000/9,813] checkpoint saved | 9.4 reviews/sec | ~10 min remaining | 0 errors
  [4,500/9,813] checkpoint saved | 9.4 reviews/sec | ~9 min remaining | 0 errors
  [5,000/9,813] checkpoint saved | 9.2 reviews/sec | ~9 min rema

In [6]:
hk_results = await extract_market('Hong Kong')


Market: Hong Kong
Reviews after filtering: 2,924
Output: /Users/juniork/Desktop/Case study/Foodpanda Analysis/v2_production/hong_kong_aspects.json
Processing reviews 0 to 2,924 (concurrency=10)...

  [500/2,924] checkpoint saved | 2.2 reviews/sec | ~18 min remaining | 0 errors
  [1,000/2,924] checkpoint saved | 1.9 reviews/sec | ~17 min remaining | 0 errors
  [1,500/2,924] checkpoint saved | 1.8 reviews/sec | ~14 min remaining | 2 errors
  [2,000/2,924] checkpoint saved | 1.7 reviews/sec | ~9 min remaining | 2 errors
  [2,500/2,924] checkpoint saved | 1.6 reviews/sec | ~4 min remaining | 2 errors
  [2,924/2,924] checkpoint saved | 1.6 reviews/sec | ~0 min remaining | 2 errors

Done: Hong Kong
  Total: 2,924 reviews
  Truncated recoveries: 71
  Errors: 2 (0.1%)
  Time: 31.4 min
  Saved to: /Users/juniork/Desktop/Case study/Foodpanda Analysis/v2_production/hong_kong_aspects.json


In [7]:
tw_results = await extract_market('Taiwan') 


Market: Taiwan
Reviews after filtering: 20,425
Output: /Users/juniork/Desktop/Case study/Foodpanda Analysis/v2_production/taiwan_aspects.json
Processing reviews 0 to 20,425 (concurrency=10)...

  [500/20,425] checkpoint saved | 2.6 reviews/sec | ~127 min remaining | 0 errors
  [1,000/20,425] checkpoint saved | 2.6 reviews/sec | ~124 min remaining | 0 errors
  [1,500/20,425] checkpoint saved | 2.7 reviews/sec | ~118 min remaining | 0 errors
  [2,000/20,425] checkpoint saved | 2.8 reviews/sec | ~111 min remaining | 0 errors
  [2,500/20,425] checkpoint saved | 2.9 reviews/sec | ~103 min remaining | 0 errors
  [3,000/20,425] checkpoint saved | 3.0 reviews/sec | ~97 min remaining | 0 errors
  [3,500/20,425] checkpoint saved | 3.1 reviews/sec | ~92 min remaining | 0 errors
  [4,000/20,425] checkpoint saved | 3.1 reviews/sec | ~88 min remaining | 0 errors
  [4,500/20,425] checkpoint saved | 3.2 reviews/sec | ~82 min remaining | 0 errors
  [5,000/20,425] checkpoint saved | 3.4 reviews/sec | ~

In [10]:
my_results = await extract_market('Malaysia')


Market: Malaysia
Reviews after filtering: 61,646
Output: /Users/juniork/Desktop/Case study/Foodpanda Analysis/v2_production/malaysia_aspects.json
Processing reviews 0 to 61,646 (concurrency=10)...

  [500/61,646] checkpoint saved | 8.0 reviews/sec | ~128 min remaining | 0 errors
  [1,000/61,646] checkpoint saved | 7.9 reviews/sec | ~128 min remaining | 0 errors
  [1,500/61,646] checkpoint saved | 8.4 reviews/sec | ~120 min remaining | 0 errors
  [2,000/61,646] checkpoint saved | 7.9 reviews/sec | ~125 min remaining | 0 errors
  [2,500/61,646] checkpoint saved | 7.9 reviews/sec | ~124 min remaining | 0 errors
  [3,000/61,646] checkpoint saved | 7.8 reviews/sec | ~125 min remaining | 0 errors
  [3,500/61,646] checkpoint saved | 7.7 reviews/sec | ~126 min remaining | 1 errors
  [4,000/61,646] checkpoint saved | 7.5 reviews/sec | ~129 min remaining | 1 errors
  [4,500/61,646] checkpoint saved | 7.1 reviews/sec | ~134 min remaining | 1 errors
  [5,000/61,646] checkpoint saved | 6.7 reviews

In [11]:
pk_results =  await extract_market('Pakistan')


Market: Pakistan
Reviews after filtering: 128,485
Output: /Users/juniork/Desktop/Case study/Foodpanda Analysis/v2_production/pakistan_aspects.json
Processing reviews 0 to 128,485 (concurrency=10)...

  [500/128,485] checkpoint saved | 6.0 reviews/sec | ~354 min remaining | 0 errors
  [1,000/128,485] checkpoint saved | 6.6 reviews/sec | ~324 min remaining | 0 errors
  [1,500/128,485] checkpoint saved | 5.0 reviews/sec | ~427 min remaining | 0 errors
  [2,000/128,485] checkpoint saved | 5.1 reviews/sec | ~416 min remaining | 0 errors
  [2,500/128,485] checkpoint saved | 5.2 reviews/sec | ~404 min remaining | 0 errors
  [3,000/128,485] checkpoint saved | 5.5 reviews/sec | ~381 min remaining | 0 errors
  [3,500/128,485] checkpoint saved | 5.9 reviews/sec | ~353 min remaining | 0 errors
  [4,000/128,485] checkpoint saved | 6.2 reviews/sec | ~332 min remaining | 0 errors
  [4,500/128,485] checkpoint saved | 6.4 reviews/sec | ~323 min remaining | 0 errors
  [5,000/128,485] checkpoint saved |

In [12]:
mm_results = await extract_market('Myanmar') 


Market: Myanmar
Reviews after filtering: 6,670
Output: /Users/juniork/Desktop/Case study/Foodpanda Analysis/v2_production/myanmar_aspects.json
Processing reviews 0 to 6,670 (concurrency=10)...

  [500/6,670] checkpoint saved | 6.1 reviews/sec | ~17 min remaining | 0 errors
  [1,000/6,670] checkpoint saved | 7.1 reviews/sec | ~13 min remaining | 0 errors
  [1,500/6,670] checkpoint saved | 6.1 reviews/sec | ~14 min remaining | 0 errors
  [2,000/6,670] checkpoint saved | 5.8 reviews/sec | ~13 min remaining | 0 errors
  [2,500/6,670] checkpoint saved | 5.7 reviews/sec | ~12 min remaining | 0 errors
  [3,000/6,670] checkpoint saved | 5.7 reviews/sec | ~11 min remaining | 0 errors
  [3,500/6,670] checkpoint saved | 5.5 reviews/sec | ~10 min remaining | 0 errors
  [4,000/6,670] checkpoint saved | 5.4 reviews/sec | ~8 min remaining | 0 errors
  [4,500/6,670] checkpoint saved | 5.3 reviews/sec | ~7 min remaining | 0 errors
  [5,000/6,670] checkpoint saved | 5.2 reviews/sec | ~5 min remaining |

In [13]:
bd_results = await extract_market('Bangladesh')


Market: Bangladesh
Reviews after filtering: 41,083
Output: /Users/juniork/Desktop/Case study/Foodpanda Analysis/v2_production/bangladesh_aspects.json
Processing reviews 0 to 41,083 (concurrency=10)...

  [500/41,083] checkpoint saved | 10.0 reviews/sec | ~67 min remaining | 0 errors
  [1,000/41,083] checkpoint saved | 9.4 reviews/sec | ~71 min remaining | 0 errors
  [1,500/41,083] checkpoint saved | 10.2 reviews/sec | ~65 min remaining | 0 errors
  [2,000/41,083] checkpoint saved | 10.2 reviews/sec | ~64 min remaining | 0 errors
  [2,500/41,083] checkpoint saved | 10.1 reviews/sec | ~64 min remaining | 0 errors
  [3,000/41,083] checkpoint saved | 9.9 reviews/sec | ~64 min remaining | 0 errors
  [3,500/41,083] checkpoint saved | 10.1 reviews/sec | ~62 min remaining | 0 errors
  [4,000/41,083] checkpoint saved | 10.0 reviews/sec | ~62 min remaining | 0 errors
  [4,500/41,083] checkpoint saved | 9.2 reviews/sec | ~66 min remaining | 0 errors
  [5,000/41,083] checkpoint saved | 8.6 review

In [14]:
la_results = await extract_market('Laos')


Market: Laos
Reviews after filtering: 3,239
Output: /Users/juniork/Desktop/Case study/Foodpanda Analysis/v2_production/laos_aspects.json
Processing reviews 0 to 3,239 (concurrency=10)...

  [500/3,239] checkpoint saved | 9.0 reviews/sec | ~5 min remaining | 0 errors
  [1,000/3,239] checkpoint saved | 9.9 reviews/sec | ~4 min remaining | 0 errors
  [1,500/3,239] checkpoint saved | 10.4 reviews/sec | ~3 min remaining | 0 errors
  [2,000/3,239] checkpoint saved | 10.4 reviews/sec | ~2 min remaining | 0 errors
  [2,500/3,239] checkpoint saved | 10.4 reviews/sec | ~1 min remaining | 0 errors
  [3,000/3,239] checkpoint saved | 10.3 reviews/sec | ~0 min remaining | 0 errors
  [3,239/3,239] checkpoint saved | 10.0 reviews/sec | ~0 min remaining | 0 errors

Done: Laos
  Total: 3,239 reviews
  Truncated recoveries: 61
  Errors: 0 (0.0%)
  Time: 5.4 min
  Saved to: /Users/juniork/Desktop/Case study/Foodpanda Analysis/v2_production/laos_aspects.json


In [15]:
kh_results = await extract_market('Cambodia') 


Market: Cambodia
Reviews after filtering: 5,438
Output: /Users/juniork/Desktop/Case study/Foodpanda Analysis/v2_production/cambodia_aspects.json
Processing reviews 0 to 5,438 (concurrency=10)...

  [500/5,438] checkpoint saved | 8.5 reviews/sec | ~10 min remaining | 0 errors
  [1,000/5,438] checkpoint saved | 9.3 reviews/sec | ~8 min remaining | 0 errors
  [1,500/5,438] checkpoint saved | 9.0 reviews/sec | ~7 min remaining | 0 errors
  [2,000/5,438] checkpoint saved | 8.1 reviews/sec | ~7 min remaining | 0 errors
  [2,500/5,438] checkpoint saved | 7.9 reviews/sec | ~6 min remaining | 0 errors
  [3,000/5,438] checkpoint saved | 7.9 reviews/sec | ~5 min remaining | 0 errors
  [3,500/5,438] checkpoint saved | 7.8 reviews/sec | ~4 min remaining | 0 errors
  [4,000/5,438] checkpoint saved | 8.0 reviews/sec | ~3 min remaining | 0 errors
  [4,500/5,438] checkpoint saved | 8.0 reviews/sec | ~2 min remaining | 0 errors
  [5,000/5,438] checkpoint saved | 8.2 reviews/sec | ~1 min remaining | 0 e

**Extraction summary across all 9 markets:** 279,723 reviews processed. Total wall-clock time varied by market size and API latency — Singapore (9,813 reviews) completed in ~19 minutes; Pakistan (128,485 reviews) took ~700 minutes across several sessions with checkpoint resumption. Residual API errors (503 timeouts, null responses) affected fewer than 0.1% of reviews overall. The `ok_truncated` status indicates reviews where `parse_gemini_json` successfully recovered a valid result from malformed model output (the `[]\n[]` pattern described in Section 3).

---

## 5. Flatten to CSV

The extraction output is one JSON file per market, with nested aspect arrays per review. The downstream analysis notebook needs a single flat CSV — one row per (review × aspect) pair, enriched with restaurant metadata from the v1 master file. Reviews with no extracted aspects get one row with null aspect/sentiment/evidence fields so they aren't silently dropped from aggregations.

In [17]:
V1_CSV = '/Users/juniork/Desktop/Case study/Foodpanda Analysis/foodpanda_APAC_final.csv'

# All 9 extraction files in size order
markets = [
    'singapore_aspects.json',
    'hong_kong_aspects.json',
    'laos_aspects.json',
    'cambodia_aspects.json',
    'myanmar_aspects.json',
    'taiwan_aspects.json',
    'bangladesh_aspects.json',
    'malaysia_aspects.json',
    'pakistan_aspects.json',
]

# Step 1: flatten each market — one row per (review × aspect)
all_rows = []
for fname in markets:
    fpath = OUTPUT_DIR / fname
    with open(fpath) as f:
        data = json.load(f)

    print(f"Loading {fname}: {len(data):,} reviews")

    for r in data:
        base = {
            'review_idx': r['review_idx'],
            'storeId': r['storeId'],
            'country': r['country'],
            'rating': r['rating'],
            'text': r['text'],
            'createdAt': r['createdAt'],
            'parse_status': r['parse_status'],
            'rider_rating': r.get('rider'),
            'food_rating': r.get('restaurant_food'),
        }

        # If no aspects extracted, output one row with null aspect
        if not isinstance(r['parsed'], list) or len(r['parsed']) == 0:
            row = {**base, 'aspect': None, 'sentiment': None, 'evidence': None}
            all_rows.append(row)
            continue

        # Otherwise one row per aspect
        for a in r['parsed']:
            if not isinstance(a, dict):
                continue
            row = {
                **base,
                'aspect': a.get('aspect'),
                'sentiment': a.get('sentiment'),
                'evidence': a.get('evidence'),
            }
            all_rows.append(row)

flat = pd.DataFrame(all_rows)
print(f"\nFlattened: {len(flat):,} rows from {flat['review_idx'].nunique():,} unique review positions")

# Step 2: load v1 master to get restaurant metadata
print("\nLoading restaurant metadata from v1...")
v1 = pd.read_csv(V1_CSV, low_memory=False,
                 usecols=['StoreId', 'CompleteStoreName', 'FoodType', 'City', 'region',
                          'AverageRating', 'Reviewers', 'latitude', 'longitude'])

# Deduplicate to one row per StoreId (it's currently one row per review)
v1_stores = v1.drop_duplicates(subset='StoreId').reset_index(drop=True)
print(f"Unique stores: {len(v1_stores):,}")

# Step 3: left-join on storeId (StoreId in v1, storeId in flat)
merged = flat.merge(
    v1_stores,
    left_on='storeId',
    right_on='StoreId',
    how='left'
).drop(columns=['StoreId'])  # redundant after merge

# Step 4: sanity checks
print(f"\nFinal rows: {len(merged):,}")
print(f"\nReviews matched to restaurant metadata: "
      f"{merged['CompleteStoreName'].notna().sum():,} / {len(merged):,} "
      f"({merged['CompleteStoreName'].notna().mean()*100:.1f}%)")

print(f"\nRows by country:")
print(merged['country'].value_counts())

print(f"\nRows by aspect (non-null only):")
print(merged[merged['aspect'].notna()]['aspect'].value_counts())

# Step 5: save
output_path = OUTPUT_DIR / 'v2_aspects_flat.csv'
merged.to_csv(output_path, index=False)

size_mb = os.path.getsize(output_path) / (1024**2)
print(f"\nSaved: {output_path}")
print(f"File size: {size_mb:.1f} MB")

Loading singapore_aspects.json: 9,813 reviews
Loading hong_kong_aspects.json: 2,924 reviews
Loading laos_aspects.json: 3,239 reviews
Loading cambodia_aspects.json: 5,438 reviews
Loading myanmar_aspects.json: 6,670 reviews
Loading taiwan_aspects.json: 20,425 reviews
Loading bangladesh_aspects.json: 41,083 reviews
Loading malaysia_aspects.json: 61,646 reviews
Loading pakistan_aspects.json: 128,485 reviews

Flattened: 315,349 rows from 128,485 unique review positions

Loading restaurant metadata from v1...
Unique stores: 151,426

Final rows: 315,349

Reviews matched to restaurant metadata: 315,349 / 315,349 (100.0%)

Rows by country:
country
Pakistan      140403
Malaysia       69903
Bangladesh     46965
Taiwan         25815
Singapore      11410
Myanmar         7696
Cambodia        5998
Laos            3631
Hong Kong       3528
Name: count, dtype: int64

Rows by aspect (non-null only):
aspect
order_accuracy         68610
vendor_quality_flag    37566
packaging              17299
delivery_ti

---

## 6. Notes on Limitations and Validation

The extraction prompt (V6) was validated on a held-out 30-review stratified sample from Singapore (6 reviews per star rating, 1–5). I labelled the sample independently after the prompt was locked — the labels were my own judgment, not a second model's output.

**Results against human labels:**
- 70% review-level exact agreement
- F1 = 56% (precision = 43%, recall = 82%)

The lower precision reflects the LLM's tendency to surface aspects I considered too trivial for action — a borderline "rider was nice" tagged as `rider_service` where I would have left it unlabelled. Recall remained high (82%), meaning the model rarely missed aspects I flagged. For a production use case focused on surfacing issues rather than suppressing false positives, high recall is the more important metric.

> **Language coverage limitation:** The V6 prompt and validation sample are English-dominant. Markets like Bangladesh (Bengali), Pakistan (Urdu), Cambodia (Khmer), Myanmar (Burmese), and Laos (Lao) contain significant non-English review text. Gemini Flash Lite supports these languages, but extraction accuracy has not been formally validated beyond English, Malay, and Mandarin. Spot-checks will be performed market-by-market after extraction completes and documented in the README.

> **LLM hallucination at scale:** LLM extraction carries a 1–3% hallucination rate at production scale. The prompt mitigates this by requiring direct-quoted evidence for every aspect label — fabricated aspects are easier to catch when the model must point to specific text. Aggregate findings across thousands of reviews are reliable; individual aspect labels may flicker between runs. This is a known tradeoff of LLM-based extraction.

> **Survivorship bias in review data:** This dataset reflects publicly posted reviews only. Customers whose orders were cancelled or refunded before they could leave a review do not appear in this data. Customers who experienced severe issues but chose not to review are also absent. This means the analysis describes the *floor* of dissatisfaction, not the ceiling — the true rate of delivery problems is likely higher than what reviews capture.

---

## Cost Note

The total billed cost for this project was **SGD 30.98** (~USD 24), as reported by Google AI Studio. This covers the full lifecycle — 6 prompt iterations during development, the 30-review validation run, and the production extraction of 279,723 reviews across 9 markets.

A separate token-estimation script (using a `len(text) // 4` character-to-token heuristic) estimated the production extraction alone at **SGD 27.26**. The ~SGD 3.72 gap reflects development iterations, retry overhead, and connectivity tests that are not captured by the heuristic.

The extraction used the synchronous-style async API (`client.aio.models.generate_content`) with `concurrency=10`, not Google's batch API. A batch endpoint would likely reduce cost further, but the synchronous approach gave real-time progress visibility and checkpoint control — a worthwhile tradeoff for a first production run where monitoring mattered more than squeezing the last dollar.